# Self-host Qwen3-VL (video) on AMD Instinct MI300X

Turnkey notebook to serve **Qwen3-VL-30B-A3B-Instruct** (video-capable, fits one
MI300X 192GB) with an OpenAI-compatible API on **ROCm**, then caption the Track-2
sample clips from the **real video** (not just frames).

**Run on:** AMD Developer Cloud MI300X instance (ROCm 6.2+). Run the cells top to bottom.
Serving the 235B variant needs FP8/multi-GPU; the 30B-A3B MoE is the practical single-GPU pick.


## 1. Environment check


In [ ]:
import subprocess
print(subprocess.run(['rocm-smi'], capture_output=True, text=True).stdout[:2000])
print(subprocess.run(['python','-c','import torch;print("torch",torch.__version__,"| cuda(rocm):",torch.cuda.is_available())'],capture_output=True,text=True).stdout)


## 2. Install vLLM (ROCm build) + deps
If your image already has vLLM+ROCm, skip. Otherwise this pulls the ROCm wheels.


In [ ]:
# ROCm vLLM. On AMD Dev Cloud the base image usually ships torch+rocm already.
!pip -q install --upgrade vllm 'transformers>=4.49' qwen-vl-utils accelerate httpx || true
!pip -q install ffmpeg-python || true
print('deps ready')


## 3. Launch the vLLM OpenAI-compatible server
Qwen3-VL with vision+video. Adjust `--max-model-len` / `--limit-mm-per-prompt` for memory.


In [ ]:
import subprocess, time, os, socket
MODEL='Qwen/Qwen3-VL-30B-A3B-Instruct'
cmd=['vllm','serve',MODEL,'--host','0.0.0.0','--port','8000',
     '--max-model-len','32768','--limit-mm-per-prompt','image=16,video=1',
     '--tensor-parallel-size','1','--gpu-memory-utilization','0.92']
server=subprocess.Popen(cmd)
# wait for readiness
import httpx
for _ in range(120):
    try:
        if httpx.get('http://localhost:8000/v1/models',timeout=2).status_code==200:
            print('server ready'); break
    except Exception: pass
    time.sleep(5)
else: print('server not ready - check logs')


## 4. Caption the sample clips from the REAL video
Downsamples each clip and sends it as video to the local Qwen3-VL server.


In [ ]:
import httpx, base64, json, subprocess, tempfile, os
BASE='http://localhost:8000/v1'; MODEL='Qwen/Qwen3-VL-30B-A3B-Instruct'
TASKS=json.load(open('../data/sample_tasks.json')) if os.path.exists('../data/sample_tasks.json') else json.load(open('data/sample_tasks.json'))
SYS=('You are an expert video captioner. Watch the clip and return STRICT JSON with four styled captions of the SAME scene: '
     '{"formal":..,"sarcastic":..,"humorous_tech":..,"humorous_non_tech":..}. Be richly detailed and accurate; do not invent; '
     'never state race/skin or eye color; do not quote an unreadable sign; English only.')
def small_video(url):
    t=tempfile.mkdtemp(); raw=f'{t}/c.mp4'; sm=f'{t}/s.mp4'
    httpx.stream and open(raw,'wb').write(httpx.get(url,follow_redirects=True,timeout=60).content)
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i',raw,'-t','12','-vf','fps=2,scale=480:-2','-an',sm],check=True)
    return base64.b64encode(open(sm,'rb').read()).decode()
results=[]
for task in TASKS:
    b64=small_video(task['video_url'])
    content=[{'type':'text','text':'Caption this clip.'},{'type':'video_url','video_url':{'url':f'data:video/mp4;base64,{b64}'}}]
    r=httpx.post(f'{BASE}/chat/completions',json={'model':MODEL,'messages':[{'role':'system','content':SYS},{'role':'user','content':content}],'max_tokens':1200,'temperature':0.4},timeout=300)
    txt=r.json()['choices'][0]['message']['content']; caps=json.loads(txt[txt.find('{'):txt.rfind('}')+1])
    results.append({'task_id':task['task_id'],'captions':caps}); print(task['task_id'],'done')
print(json.dumps(results,indent=2)[:1500])
json.dump(results,open('qwen_video_results.json','w'),indent=2)


## 5. Point the Track-2 Docker pipeline at this server (optional)
The container speaks the OpenAI API, so just override the base URL + model:

```bash
OPENROUTER_BASE_URL=http://<this-instance-ip>:8000/v1 \
OPENROUTER_VLM_MODEL=Qwen/Qwen3-VL-30B-A3B-Instruct \
OPENROUTER_API_KEY=EMPTY \
python -m app.main
```

This is the **'Use of AMD Platforms'** story: Qwen3-VL served on MI300X + ROCm,
captioning real video the hosted APIs cannot ingest.
